# Your first Polars workflow
Run each cell in order. The data is synthetic.

In [2]:
from pathlib import Path

import polars as pl

root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
donors = pl.read_csv(root / 'data/raw/donors.csv', try_parse_dates=True)
donors.head()

donor_id,join_date,state_group,age_band,membership_status,acquisition_source,email_opt_in,mail_opt_in,household_member,volunteer,event_attendance_count
str,date,str,str,str,str,bool,bool,bool,bool,i64
"""D000001""",2012-09-19,"""NC""","""50-64""","""Active""","""Website""",true,true,true,true,2
"""D000002""",2020-02-24,"""GA""","""50-64""","""Active""","""Direct mail""",true,true,false,false,0
"""D000003""",2016-12-10,"""TN""","""65+""","""Lapsed""","""Direct mail""",false,true,false,false,0
"""D000004""",2022-04-25,"""PA""","""35-49""","""Prospect""","""Friend""",true,true,false,false,1
"""D000005""",2013-07-02,"""PA""","""50-64""","""Lapsed""","""Magazine""",true,true,false,false,1


In [3]:
print(donors.shape)
print(donors.schema)
donors.null_count()

(3000, 11)
Schema({'donor_id': String, 'join_date': Date, 'state_group': String, 'age_band': String, 'membership_status': String, 'acquisition_source': String, 'email_opt_in': Boolean, 'mail_opt_in': Boolean, 'household_member': Boolean, 'volunteer': Boolean, 'event_attendance_count': Int64})


donor_id,join_date,state_group,age_band,membership_status,acquisition_source,email_opt_in,mail_opt_in,household_member,volunteer,event_attendance_count
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,223,0,0,0,0,0,0,0


In [4]:
summary = (donors.filter(pl.col('membership_status') == 'Active')
           .group_by('state_group').agg(pl.len().alias('active_donors'))
           .sort('active_donors', descending=True))
summary

state_group,active_donors
str,u32
"""Other""",366
"""NC""",301
"""VA""",238
"""PA""",207
"""NY""",164
"""TN""",143
"""OH""",122
"""GA""",101


## Practice
Count donors by acquisition source, calculate the email opt-in rate, and save one result under `outputs/`.

In [5]:
from pathlib import Path
import polars as pl

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
donors = pl.read_csv(root / "data/raw/donors.csv", try_parse_dates=True)
counts = donors.group_by("membership_status").agg(pl.len().alias("donors"))
print(counts.sort("membership_status"))

output_dir = root / "outputs"
output_dir.mkdir(exist_ok=True)
counts.write_csv(output_dir / "lesson_2_membership_counts.csv")

shape: (3, 2)
┌───────────────────┬────────┐
│ membership_status ┆ donors │
│ ---               ┆ ---    │
│ str               ┆ u32    │
╞═══════════════════╪════════╡
│ Active            ┆ 1642   │
│ Lapsed            ┆ 812    │
│ Prospect          ┆ 546    │
└───────────────────┴────────┘


In [ ]:
from pathlib import Path
import polars as pl

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
donors = pl.read_csv(root / "data/raw/donors.csv", try_parse_dates=True)
counts = donors.group_by("acquisition_source").agg(pl.len().alias("donors"))
print(counts.sort("acquisition_source", descending=True))

#each row represents the count of people who used a particular
#aquisition source

#write_csv stores the csv in a specified output directory
#filter only keeps rows that meet a specific criteria
#group_by groups rows that meet a specific criteria together.
# if used without agg, it returns a frame with that column 
#sorted if it is categorical
#agg allows you to calculate stuff for each group, such as the
#sum or mean.

output_dir = root / "outputs"
output_dir.mkdir(exist_ok=True)
counts.write_csv(output_dir/"lesson_2_acquisition_counts.csv")

shape: (6, 2)
┌────────────────────┬────────┐
│ acquisition_source ┆ donors │
│ ---                ┆ ---    │
│ str                ┆ u32    │
╞════════════════════╪════════╡
│ Website            ┆ 894    │
│ Unknown            ┆ 277    │
│ Magazine           ┆ 554    │
│ Friend             ┆ 321    │
│ Event              ┆ 406    │
│ Direct mail        ┆ 548    │
└────────────────────┴────────┘


In [14]:
from pathlib import Path
import polars as pl

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
donors = pl.read_csv(root / "data/raw/donors.csv", try_parse_dates=True)
counts = donors.group_by("email_opt_in").agg(pl.col("email_opt_in").len().alias("donors"))
print(counts.sort("email_opt_in"))



shape: (2, 2)
┌──────────────┬────────┐
│ email_opt_in ┆ donors │
│ ---          ┆ ---    │
│ bool         ┆ u32    │
╞══════════════╪════════╡
│ false        ┆ 843    │
│ true         ┆ 2157   │
└──────────────┴────────┘


In [16]:
import polars as pl

# 1. Create a sample DataFrame
df = pl.DataFrame({
    "condition": ["Flu", "Flu", "Covid", "Flu", "Covid", "Cold"],
    "opted_in_email": [True, False, True, True, False, True]  # Boolean column
})

# 2. Aggregate counts and opt-ins
result = df.group_by("condition").agg([
    pl.len().alias("total_count"),
    pl.col("opted_in_email").sum().alias("opted_in_count")
])

print(result)


shape: (3, 3)
┌───────────┬─────────────┬────────────────┐
│ condition ┆ total_count ┆ opted_in_count │
│ ---       ┆ ---         ┆ ---            │
│ str       ┆ u32         ┆ u32            │
╞═══════════╪═════════════╪════════════════╡
│ Flu       ┆ 3           ┆ 2              │
│ Cold      ┆ 1           ┆ 1              │
│ Covid     ┆ 2           ┆ 1              │
└───────────┴─────────────┴────────────────┘


In [18]:
from pathlib import Path
import polars as pl

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
donors = pl.read_csv(root / "data/raw/donors.csv", try_parse_dates=True)
counts = donors.group_by("acquisition_source")
print(counts.head())

#each row represents the count of people who used a particular
#aquisition source

#write_csv stores the csv in a specified output directory
#filter only keeps rows that meet a specific criteria
#



shape: (30, 11)
┌───────────┬──────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ acquisiti ┆ donor_id ┆ join_date ┆ state_gro ┆ … ┆ mail_opt_ ┆ household ┆ volunteer ┆ event_att │
│ on_source ┆ ---      ┆ ---       ┆ up        ┆   ┆ in        ┆ _member   ┆ ---       ┆ endance_c │
│ ---       ┆ str      ┆ date      ┆ ---       ┆   ┆ ---       ┆ ---       ┆ bool      ┆ ount      │
│ str       ┆          ┆           ┆ str       ┆   ┆ bool      ┆ bool      ┆           ┆ ---       │
│           ┆          ┆           ┆           ┆   ┆           ┆           ┆           ┆ i64       │
╞═══════════╪══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ Friend    ┆ D000004  ┆ 2022-04-2 ┆ PA        ┆ … ┆ true      ┆ false     ┆ false     ┆ 1         │
│           ┆          ┆ 5         ┆           ┆   ┆           ┆           ┆           ┆           │
│ Friend    ┆ D000006  ┆ 2010-09-2 ┆ NY        ┆ … ┆ true      ┆ false     

In [1]:
from fundraising_analytics.load_data import load_tables
from fundraising_analytics.metrics import deduplicate_gifts

tables = load_tables()
gifts = deduplicate_gifts(tables["gifts"])
before_rows = gifts.height
before_revenue = gifts["gift_amount"].sum()

joined = gifts.join(tables["donors"], on="donor_id", how="left", validate="m:1")
print("Rows before / after:", before_rows, joined.height)
print("Revenue before / after:", before_revenue, joined["gift_amount"].sum())




Rows before / after: 8988 8988
Revenue before / after: 6242385.0 6242385.0


In [2]:
import math

tables = load_tables()
raw_gifts = tables["gifts"]
gifts = deduplicate_gifts(raw_gifts)

joined = gifts.join(tables["donors"], on="donor_id", how="left", validate="m:1")

assert joined.height == gifts.height, "Join changed row count"
assert math.isclose(joined["gift_amount"].sum(), gifts["gift_amount"].sum()), "Join changed revenue"

orphans = gifts.join(tables["donors"], on="donor_id", how="anti")
print(f"Dedup removed {raw_gifts.height - gifts.height} rows "
      f"({raw_gifts['gift_amount'].sum() - gifts['gift_amount'].sum():,.2f} revenue)")
print(f"Gifts with no matching donor: {orphans.height} "
      f"({orphans['gift_amount'].sum():,.2f} revenue)")

Dedup removed 12 rows (2,385.00 revenue)
Gifts with no matching donor: 0 (0.00 revenue)


In [ ]:
import polars as pl
tables=load_tables()
df=tables["gifts"]
dupes = df.filter(pl.col("gift_id").is_duplicated())
print(dupes)
#question: would it be acceptable to delete if there is also an issue with the solicitation code?


shape: (24, 10)
┌──────────┬──────────┬────────────┬────────────┬───┬────────────┬────────┬────────────┬───────────┐
│ gift_id  ┆ donor_id ┆ gift_date  ┆ gift_amoun ┆ … ┆ solicitati ┆ online ┆ recurring_ ┆ soft_cred │
│ ---      ┆ ---      ┆ ---        ┆ t          ┆   ┆ on_code    ┆ ---    ┆ gift       ┆ it        │
│ str      ┆ str      ┆ date       ┆ ---        ┆   ┆ ---        ┆ bool   ┆ ---        ┆ ---       │
│          ┆          ┆            ┆ f64        ┆   ┆ str        ┆        ┆ bool       ┆ bool      │
╞══════════╪══════════╪════════════╪════════════╪═══╪════════════╪════════╪════════════╪═══════════╡
│ G0000255 ┆ D002105  ┆ 2023-09-17 ┆ 750.0      ┆ … ┆ null       ┆ false  ┆ false      ┆ false     │
│ G0000317 ┆ D000124  ┆ 2025-12-09 ┆ 75.0       ┆ … ┆ EOY2025    ┆ false  ┆ true       ┆ false     │
│ G0000633 ┆ D002425  ┆ 2024-06-04 ┆ 250.0      ┆ … ┆ WEB2024S   ┆ true   ┆ false      ┆ false     │
│ G0001506 ┆ D000827  ┆ 2023-12-26 ┆ 100.0      ┆ … ┆ EOY2023    ┆ false  ┆

In [2]:
import polars as pl

tables=load_tables()
df=tables["gifts"]

# Deduplicate based on the "id" column (keeps the first occurrence by default)
#would you use any for max randomness?
deduped_df = df.unique(subset="gift_id", keep="first")
print(deduped_df)
#check that deduped is shorter
print (f"length before: {df.height}, length after: {deduped_df.height}")

dupes = df.filter(pl.col("gift_id").is_duplicated())
print(dupes.height)

assert dupes.height/2+deduped_df.height==df.height
#this assumes things are only dupes, not triplicates though
#is there a way to check for that?

NameError: name 'load_tables' is not defined

In [1]:
import polars as pl

tables=load_tables()
df=tables["large_gift_attribution"]

NameError: name 'load_tables' is not defined